
---

## 📚 Sobre este Material

Este material ha sido diseñado con el propósito de **capacitar, actualizar y practicar** conceptos fundamentales de Markdown en Jupyter Notebook. Es una herramienta pensada para facilitar el aprendizaje y la documentación efectiva de proyectos de análisis de datos y ciencia de datos.

### 🤝 Compartir y Colaborar

Este contenido es **libre para compartir, revisar, divulgar y mejorar**. Se promueve activamente su distribución en la comunidad para que más personas puedan beneficiarse y contribuir a su mejora continua. Tu feedback y sugerencias son siempre bienvenidos.

### 👨‍💻 Autor

**Andrés Muñoz**  
*AI & Data Strategy Leader passionate about NLP, LLMs, and MLOps. Driving innovation with data*

- 💼 LinkedIn: [in/amms1989](https://linkedin.com/in/amms1989)
- 🐙 GitHub: [https://github.com/anguihero](https://github.com/anguihero)

---

# Sesión 14: Optimización de Hiperparámetros

**Autor:** anmmunozsa@outlook.es · Material de código abierto para compartir y aprender colectivamente.

## 🎯 Objetivo de la sesión
Afinar el pipeline de la Sesión 13 (`load_diabetes`) usando Grid Search, Random Search y Optimización Bayesiana (Optuna), comparando su eficiencia.

## 🗺️ Tabla de Contenido
1. [Introducción: parámetros vs. hiperparámetros](#intro)
2. [Preparar el pipeline base](#base)
3. [Grid Search](#grid)
4. [Random Search](#random)
5. [Optimización Bayesiana con Optuna](#optuna)
6. [Comparación final](#comparacion)
7. [Ejemplos de aplicación real](#aplicaciones)
8. [Retos de práctica](#retos)


<a id="intro"></a>
## 1. Introducción (para dummies)

- **Parámetros:** los aprende el modelo solo durante `.fit()` (ej. los coeficientes de una regresión).
- **Hiperparámetros:** los defines tú **antes** de entrenar (ej. `n_estimators`, `max_depth`, `alpha`).

Hoy automatizamos la búsqueda del mejor hiperparámetro con 3 estrategias distintas, todas aplicadas sobre el mismo pipeline para comparar en igualdad de condiciones.

<a id="base"></a>
## 2. Preparar el Pipeline Base

Usamos un `RandomForestRegressor` dentro de un `Pipeline` (con la rama numérica de `ColumnTransformer` vista en la Sesión 13), ya que tiene varios hiperparámetros interesantes para afinar.

In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
import pandas as pd
import time

datos = load_diabetes(as_frame=True)
X, y = datos.data, datos.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

pipeline_base = Pipeline(steps=[
    ("preprocesador", ColumnTransformer([("num", StandardScaler(), X.columns.tolist())])),
    ("modelo", RandomForestRegressor(random_state=42)),
])

print("R² sin tuning (valores por defecto):", pipeline_base.fit(X_train, y_train).score(X_test, y_test))

<a id="grid"></a>
## 3. Grid Search

### 🔬 Teoría técnica
`GridSearchCV` prueba **todas** las combinaciones posibles de una grilla de hiperparámetros, evaluando cada una con validación cruzada. Es exhaustivo pero su costo crece muy rápido (`combinaciones x folds` entrenamientos).

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "modelo__n_estimators": [100, 200],
    "modelo__max_depth": [4, 8, None],
    "modelo__max_features": ["sqrt", "log2"],
}

inicio = time.time()
grid_search = GridSearchCV(pipeline_base, param_grid, cv=5, scoring="r2", n_jobs=-1)
grid_search.fit(X_train, y_train)
tiempo_grid = time.time() - inicio

print("Mejor combinación:", grid_search.best_params_)
print("Mejor R² en CV:", grid_search.best_score_)
print("R² en prueba:", grid_search.best_estimator_.score(X_test, y_test))
print(f"Tiempo: {tiempo_grid:.1f} segundos ({len(param_grid['modelo__n_estimators']) * len(param_grid['modelo__max_depth']) * len(param_grid['modelo__max_features'])} combinaciones x 5 folds)")

### 🧠 Resumen para dummies
Grid Search es "probar TODO lo que está en el menú" — garantiza encontrar la mejor combinación dentro de la grilla que definiste, pero si la grilla es grande, tardará mucho.

<a id="random"></a>
## 4. Random Search

### 🔬 Teoría técnica
`RandomizedSearchCV` prueba un número fijo (`n_iter`) de combinaciones **aleatorias** dentro de los rangos definidos, en vez de probarlas todas — un muestreo estocástico del espacio de búsqueda que suele encontrar buenas combinaciones mucho más rápido.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

param_distributions = {
    "modelo__n_estimators": randint(50, 400),
    "modelo__max_depth": [3, 4, 6, 8, 12, None],
    "modelo__max_features": ["sqrt", "log2", None],
    "modelo__min_samples_split": randint(2, 10),
}


In [ ]:
inicio = time.time()
random_search = RandomizedSearchCV(
    pipeline_base, param_distributions, n_iter=20, cv=5,
    scoring="r2", n_jobs=-1, random_state=42,
)
random_search.fit(X_train, y_train)
tiempo_random = time.time() - inicio

print("Mejor combinación:", random_search.best_params_)
print("Mejor R² en CV:", random_search.best_score_)
print("R² en prueba:", random_search.best_estimator_.score(X_test, y_test))
print(f"Tiempo: {tiempo_random:.1f} segundos (20 combinaciones x 5 folds)")


### 🧠 Resumen para dummies
Random Search es "probar 20 combinaciones al azar en vez de las 12 exactas del menú" — con espacios de búsqueda grandes, suele encontrar algo casi igual de bueno en mucho menos tiempo.

<a id="optuna"></a>
## 5. Optimización Bayesiana con Optuna

### 🔬 Teoría técnica
Optuna no prueba combinaciones a ciegas: construye un modelo probabilístico de qué combinaciones son prometedoras **basándose en los resultados de intentos anteriores**, y concentra la búsqueda ahí. Suele necesitar menos iteraciones para llegar a un resultado similar o mejor.

In [ ]:
# En Colab, si no está instalado: !pip install -q optuna
import optuna
from sklearn.model_selection import cross_val_score

optuna.logging.set_verbosity(optuna.logging.WARNING)

def objetivo(trial):
    n_estimators = trial.suggest_int("n_estimators", 50, 400)
    max_depth = trial.suggest_categorical("max_depth", [3, 4, 6, 8, 12, None])
    max_features = trial.suggest_categorical("max_features", ["sqrt", "log2", None])

    modelo = Pipeline(steps=[
        ("preprocesador", ColumnTransformer([("num", StandardScaler(), X.columns.tolist())])),
        ("modelo", RandomForestRegressor(
            n_estimators=n_estimators, max_depth=max_depth,
            max_features=max_features, random_state=42,
        )),
    ])
    return cross_val_score(modelo, X_train, y_train, cv=5, scoring="r2").mean()

inicio = time.time()
estudio = optuna.create_study(direction="maximize")
estudio.optimize(objetivo, n_trials=20)
tiempo_optuna = time.time() - inicio


In [ ]:
print("Mejores hiperparámetros:", estudio.best_params)
print("Mejor R² en CV:", estudio.best_value)
print(f"Tiempo: {tiempo_optuna:.1f} segundos (20 pruebas)")


### 🧠 Resumen para dummies
Optuna "aprende de sus propios intentos": después de las primeras pruebas, deja de explorar zonas que claramente no funcionan y se enfoca en las prometedoras.

<a id="comparacion"></a>
## 6. Comparación Final

| Estrategia | Fortaleza | Debilidad |
|---|---|---|
| Grid Search | Exhaustivo, garantiza el óptimo de la grilla | Costoso, no escala con muchos hiperparámetros |
| Random Search | Eficiente, cubre espacios grandes | Puede pasar por alto el óptimo exacto |
| Optuna (Bayesiana) | Aprende de iteraciones previas, muy eficiente | Más compleja de configurar, requiere una librería extra |

### 🧠 Resumen para dummies
Grid si tienes pocas combinaciones y tiempo de sobra. Random si tienes muchas combinaciones posibles. Bayesiana (Optuna) si quieres lo mejor de ambos mundos y no te importa instalar una librería adicional.

In [ ]:
comparacion_final = pd.DataFrame([
    {"estrategia": "Grid Search", "mejor_r2_cv": grid_search.best_score_, "tiempo_seg": tiempo_grid},
    {"estrategia": "Random Search", "mejor_r2_cv": random_search.best_score_, "tiempo_seg": tiempo_random},
    {"estrategia": "Optuna (Bayesiana)", "mejor_r2_cv": estudio.best_value, "tiempo_seg": tiempo_optuna},
])
comparacion_final

## 🔎 Laboratorio de profundización: objetivo, presupuesto y espacio de búsqueda

Tuning optimiza una estimación de desempeño:

$$\theta^*=\arg\max_{\theta\in\Theta}\frac{1}{K}\sum_{k=1}^{K}Score_k(\theta)$$

`θ` son hiperparámetros; `Θ`, el espacio de búsqueda. El modelo sigue minimizando su propia pérdida durante cada ajuste. La búsqueda externa compara configuraciones mediante validación cruzada.


In [ ]:
# Paso 1: calcular antes el costo de una grilla
from math import prod

param_grid_demo = {
    "modelo__n_estimators": [50, 100, 200],
    "modelo__max_depth": [3, 6, None],
    "modelo__min_samples_leaf": [1, 2, 5],
}
combinaciones = prod(len(v) for v in param_grid_demo.values())
fits = combinaciones * 5
print({"combinaciones": combinaciones, "ajustes_con_cv_5": fits})


In [ ]:
# Paso 2: revisar estabilidad, no solo el mejor promedio
resultados_grid = pd.DataFrame(grid_search.cv_results_)
columnas = [
    "rank_test_score", "mean_test_score", "std_test_score",
    "mean_fit_time", "params",
]
resultados_grid[columnas].sort_values("rank_test_score").head()


### Hiperparámetros de la búsqueda

- Grid: `param_grid`, `scoring`, `cv`, `refit`, `n_jobs`.
- Random: `param_distributions`, `n_iter`, `random_state`.
- Optuna: `n_trials`, sampler, pruner y rangos `suggest_*`.

Usa escalas logarítmicas para tasas y regularización, distribuciones enteras para conteos y fija semillas. Después del tuning, evalúa **una sola vez** en test. Optimizar repetidamente contra test produce sobreajuste al conjunto de prueba.


<a id="aplicaciones"></a>
## 7. Ejemplos de Aplicación en el Mundo Real

- Equipos de ML deciden cuánto tiempo de cómputo (y por tanto, costo en la nube) invertir en tuning antes de llevar un modelo a producción.
- Competencias de Kaggle usan intensivamente Optuna para exprimir los últimos puntos porcentuales de una métrica.

<a id="retos"></a>
## 8. Retos de Práctica

### 🥉 Reto Básico
Usa `GridSearchCV` para afinar solo `n_estimators` (ej. [50, 100, 200]) de un `RandomForestRegressor` simple (sin pipeline) sobre `load_diabetes`.

In [ ]:
# Tu solución al Reto Básico aquí


### 🥈 Reto Medio
Usa `RandomizedSearchCV` con `n_iter=30` sobre un espacio de búsqueda más amplio que incluya también `min_samples_leaf`, y compara el tiempo de ejecución y el resultado contra el Reto Básico.

In [ ]:
# Tu solución al Reto Medio aquí


### 🥇 Reto Avanzado
Usa Optuna para optimizar un `GradientBoostingRegressor` (en vez de Random Forest) dentro del pipeline completo, con al menos 3 hiperparámetros (`n_estimators`, `learning_rate`, `max_depth`). Reporta en una tabla final el mejor R² y tiempo de las 3 estrategias sobre este nuevo modelo.

In [ ]:
# Tu solución al Reto Avanzado aquí
